# Manual turn-ban path check (AequilibraE vs NetworkX)

This notebook does one focused exercise:
1. Sample 1000 OD pairs
2. Compute baseline AequilibraE paths
3. Pick one random consecutive 2-link sequence (with directions) per path
4. Add all selected sequences as turn bans

In [1]:
from pathlib import Path
from collections import defaultdict

import networkx as nx
import numpy as np
import pandas as pd

from aequilibrae import Project
from tqdm.auto import tqdm

# ---- USER INPUTS ----
# MODEL_PATH = Path(r"D:\release\Sample models\new_chicago\chicago_sample_model")
# MODEL_PATH = Path(r"D:\release\coquimbo")
MODEL_PATH = Path(r"D:\release\Arkansas\model")
MODE = "c"
COST_FIELD = "distance"
SAMPLE_SIZE = 1
SEED = 42
ALLOW_UTURNS = False
from time import perf_counter

rng = np.random.default_rng(SEED)

D:\src\aequilibrae\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
def sample_od_pairs(centroids, sample_size, rng):
    if len(centroids) < 2:
        raise ValueError("Need at least 2 centroids")

    origins = rng.choice(centroids, size=sample_size, replace=True)
    destinations = rng.choice(centroids, size=sample_size, replace=True)

    same = origins == destinations
    while np.any(same):
        destinations[same] = rng.choice(centroids, size=int(np.sum(same)), replace=True)
        same = origins == destinations

    return [(int(o), int(d)) for o, d in zip(origins, destinations)]

In [3]:
project = Project.from_path(MODEL_PATH)
project.upgrade()
project.network.build_graphs(modes=[MODE])
graph = project.network.graphs[MODE]
graph.set_graph(COST_FIELD)
graph.set_skimming(COST_FIELD)
graph.set_blocked_centroid_flows(False)

centroids = np.asarray(graph.centroids, dtype=np.int64)
if centroids.size == 0:
    raise RuntimeError("No centroids found in graph")

D:\src\aequilibrae\aequilibrae\paths\graph.py:251: UserWarning: Found centroids not present in the graph!
[770]
  build_compressed_graph(self, remove_dead_ends)


In [4]:
before = - perf_counter()
_ = graph.compute_skims(64)
before += perf_counter()

5117/6448                                         :  80%|███████▉  | 5148/6448 [00:10<00:03, 408.41it/s]

In [5]:
before

13.727767099999937

In [6]:
od_pairs = sample_od_pairs(centroids, SAMPLE_SIZE, rng)
selected_turns = []
rows_for_ban_selection = []
for orig, dest in tqdm(od_pairs, desc="AEQ baseline paths", total=len(od_pairs)):
    res = graph.compute_path(orig, dest)
    if res.path_nodes is None or res.path_nodes.shape[0] < 5:  # Need at least one interior node triple
        continue

    nodes = [int(x) for x in res.path_nodes]

    idx = int(rng.integers(1, len(nodes) - 2))
    turn = (nodes[idx - 1], nodes[idx], nodes[idx + 1])
    selected_turns.append(turn)

    rows_for_ban_selection.append([orig, dest, turn[0], turn[1], turn[2]])

cols = ["orig", "dest", "from_node", "via_node", "to_node"]
selected_turns_df = pd.DataFrame(rows_for_ban_selection, columns=cols)

selected_turns_df.drop_duplicates(subset=["from_node", "via_node", "to_node"], inplace=True)
turn_bans = selected_turns_df.assign(penalty=np.nan)

if turn_bans.empty:
    raise RuntimeError("No valid node-based turns were collected for bans")

graph.set_turn_restrictions(turn_bans, allow_path_uturns=ALLOW_UTURNS)

print(f"Sampled OD pairs: {len(od_pairs)}")
print(f"Successful baseline paths: {selected_turns_df.shape[0]}")
print(f"Unique prohibited turns added: {len(turn_bans)}")

AEQ baseline paths: 100%|██████████| 1/1 [00:00<00:00, 20.83it/s]


Sampled OD pairs: 1
Successful baseline paths: 1
Unique prohibited turns added: 1


In [7]:
after = - perf_counter()
_ = graph.compute_skims(64)
after += perf_counter()

In [8]:
before, after

(13.727767099999937, 12.218073900000036)

In [9]:
project.close()
print("Project closed")

Project closed
